# Fake Jobs – Exp 1: TabPFN-unsup & FoMo-OD
- TabPFN-Unsupervised (Kontext <= 3000, ohne Labels)
- FoMo-OD zero-shot (Features -> 100, Kontext <= 5000)

In [ ]:
import sys, glob, zipfile, time
import numpy as np
import pandas as pd
import torch
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc
from sklearn.preprocessing import QuantileTransformer
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.unsupervised import TabPFNUnsupervisedModel

## Daten & Split
- Outlier = `fraudulent == 1`; 70/30 stratifiziert (seed 42)

In [2]:
df = pd.read_csv("../../data/preprocessed/cleaned_fake_jobs.csv")
y = df["fraudulent"].values
X = df.drop(columns=["row_id", "fraudulent"]).values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print("train", X_train.shape, "test", X_test.shape, "test outlier rate", round(y_test.mean(), 4))

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("fake_jobs_experiment_1")

train (12516, 39) test (5364, 39) test outlier rate 0.0485


/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/fake_job_notebooks/exp1/../../mlruns/135783001714284893', creation_time=1780130098860, experiment_id='135783001714284893', last_update_time=1780130098860, lifecycle_stage='active', name='fake_jobs_experiment_1', tags={}, trace_location=None, workspace='default'>

## TabPFN-Unsupervised
- Kontext = stratifizierte Train-Teilmenge <= 3000 (ohne Labels)
- API ggf. anpassen; Vorzeichen wird automatisch korrigiert (AUC < 0.5 -> flip)

In [ ]:
rng = np.random.RandomState(42)
idx = rng.choice(len(X_train), size=min(3000, len(X_train)), replace=False)
X_ctx = X_train[idx]

clf = TabPFNClassifier()
reg = TabPFNRegressor()
model = TabPFNUnsupervisedModel(tabpfn_clf=clf, tabpfn_reg=reg)

t0 = time.perf_counter()
model.fit(torch.from_numpy(X_ctx))
scores = np.asarray(model.outliers(torch.from_numpy(X_test))).ravel().astype(float)
runtime = time.perf_counter() - t0

auroc = roc_auc_score(y_test, scores)
if auroc < 0.5:
    scores = -scores
    auroc = roc_auc_score(y_test, scores)
ap = average_precision_score(y_test, scores)
prec, rec, _ = precision_recall_curve(y_test, scores)
auprc = auc(rec, prec)

with mlflow.start_run(run_name="tabpfn_unsup"):
    mlflow.log_param("context_size", len(X_ctx))
    mlflow.log_metric("average_precision", ap)
    mlflow.log_metric("auprc", auprc)
    mlflow.log_metric("auc_roc", auroc)
    mlflow.log_metric("runtime_s", runtime)
print(f"tabpfn_unsup: AP={ap:.4f} AUPRC={auprc:.4f} AUC={auroc:.4f} time={runtime:.1f}s")

## FoMo-OD (zero-shot)
- `FoMo0DHub` aus `../../FoMo-0D`, Gewichte aus `ckpt.zip`
- Features per QuantileTransform + Zero-Padding auf genau 100
- Kontext = Train-Subsample <= 5000 (Originalverteilung); Score = Softmax-P(Klasse 1)

In [4]:
sys.path.insert(0, "../../FoMo-0D")
from fomo_hub import FoMo0DHub

ckpts = glob.glob("../../FoMo-0D/ckpt/**/best.ckpt", recursive=True)
if not ckpts:
    with zipfile.ZipFile("../../FoMo-0D/ckpt.zip") as z:
        z.extractall("../../FoMo-0D")
    ckpts = glob.glob("../../FoMo-0D/ckpt/**/best.ckpt", recursive=True)
ckpt_path = ckpts[0]

device = "cuda" if torch.cuda.is_available() else "cpu"
fomo = FoMo0DHub(num_features=100, emsize=256, nhid=512, nlayers=4, nhead=4, num_R=500)
state = torch.load(ckpt_path, map_location=device)["state_dict"]
state = {k.replace("model.", "", 1): v for k, v in state.items()}
missing, unexpected = fomo.model.load_state_dict(state, strict=False)
print("missing:", len(missing), "unexpected:", len(unexpected))
fomo = fomo.to(device).eval()

using vanilla + router
last_layer_no_R=True, is_final_layer=False
using vanilla + router
last_layer_no_R=True, is_final_layer=False
using vanilla + router
last_layer_no_R=True, is_final_layer=False
using vanilla + router
last_layer_no_R=True, is_final_layer=True
Initialized decoder for standard with (None, 2)  and nout 2
missing: 0 unexpected: 0


In [ ]:
cidx = rng.choice(len(X_train), size=min(5000, len(X_train)), replace=False)
X_ctx = X_train[cidx]

qt = QuantileTransformer(output_distribution="normal", random_state=42, n_quantiles=min(1000, len(X_ctx)))
X_ctx_t = qt.fit_transform(X_ctx)
X_test_t = qt.transform(X_test)
pad = 100 - X_ctx_t.shape[1]
X_ctx_p = np.pad(X_ctx_t, ((0, 0), (0, pad)), constant_values=0.0).astype(np.float32)
X_test_p = np.pad(X_test_t, ((0, 0), (0, pad)), constant_values=0.0).astype(np.float32)

train_x = torch.from_numpy(X_ctx_p).unsqueeze(1).to(device)  # (ctx, 1, 100)
t0 = time.perf_counter()
probs = []
with torch.no_grad():
    for s in range(0, len(X_test_p), 256):
        chunk = torch.from_numpy(X_test_p[s:s + 256]).unsqueeze(1).to(device)  # (n, 1, 100)
        logits = fomo(train_x, chunk).squeeze(1)  # (n, 2)
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
scores = np.concatenate(probs).astype(float)
runtime = time.perf_counter() - t0

auroc = roc_auc_score(y_test, scores)
if auroc < 0.5:
    scores = -scores
    auroc = roc_auc_score(y_test, scores)
ap = average_precision_score(y_test, scores)
prec, rec, _ = precision_recall_curve(y_test, scores)
auprc = auc(rec, prec)

with mlflow.start_run(run_name="fomo_od"):
    mlflow.log_param("context_size", len(X_ctx))
    mlflow.log_metric("average_precision", ap)
    mlflow.log_metric("auprc", auprc)
    mlflow.log_metric("auc_roc", auroc)
    mlflow.log_metric("runtime_s", runtime)
print(f"fomo_od: AP={ap:.4f} AUPRC={auprc:.4f} AUC={auroc:.4f} time={runtime:.1f}s")